In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import os
from pathlib import Path

notebook_path = "/u/skarmakar1/version_check/llm_steering-main/sk"
sys.path.append(notebook_path)

In [3]:
import torch
import numpy as np

from inversion_utils import *
import pickle
from sklearn.model_selection import train_test_split

In [4]:
SEED = 0
# SEED = 1

torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
np.random.seed(SEED)

torch.backends.cudnn.benchmark = True 
torch.backends.cuda.matmul.allow_tf32 = True

LLM = namedtuple('LLM', ['language_model', 'tokenizer', 'processor', 'name', 'model_type'])

In [5]:
model_type = 'llama'
MODEL_VERSION='3.1'
MODEL_SIZE='8B'

# model_type = 'gemma'
# MODEL_VERSION='3'
# MODEL_SIZE='1B'
# MODEL_SIZE='12B'

# model_type = 'qwen'
# MODEL_VERSION='3'
# MODEL_SIZE='4B'
# # MODEL_SIZE='8B'

llm = select_llm(model_type, MODEL_VERSION=MODEL_VERSION, MODEL_SIZE=MODEL_SIZE)

Loading meta-llama/Meta-Llama-3.1-8B-Instruct


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [6]:
hidden_layers = list(range(-llm.language_model.config.num_hidden_layers+1, 0))
print(hidden_layers)

[-31, -30, -29, -28, -27, -26, -25, -24, -23, -22, -21, -20, -19, -18, -17, -16, -15, -14, -13, -12, -11, -10, -9, -8, -7, -6, -5, -4, -3, -2, -1]


Training

In [7]:
all_p_langs = ['python', 'javascript', 'java', 'c++', 'c#', 'c', 'typescript', 'php', 'golang', 'rust', 'swift', 'kotlin', 'r', 'ruby', 'dart', 'matlab', 'assembly', 'vba', 'shell', 'delphi']

In [8]:
orig_p_lang = 'python'
target_p_lang = 'c'
# test_set = ['javascript',]
# test_set = ['php',]
test_set = ['c++',]

source_shift = True
# source_shift = False

other_p_langs = [element for element in all_p_langs if element not in [orig_p_lang, target_p_lang] + test_set]

path = '../all_gitignore/xRFM/code_llama/'

X = read_single_translate(llm, orig_p_lang, other_p_langs, path, source_shift=source_shift, hidden_layers=hidden_layers)
Y = read_single_translate(llm, target_p_lang, other_p_langs, path, source_shift=source_shift, hidden_layers=hidden_layers)

Source Constant
Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 8
forward_batch_size   : 16
M_batch_size         : 2048
n_components         : 1

Using xRFM library for RFM-based steering vector extraction
Detector found
Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 8
forward_batch_size   : 16
M_batch_size         : 2048
n_components         : 1

Using xRFM library for RFM-based steering vector extraction
Detector found
Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]

Controller hyperparameter

In [9]:
print(X[-2].shape)
print(Y[-2].shape)

torch.Size([17, 4096])
torch.Size([17, 4096])


In [10]:
big_X = torch.vstack(list(X.values())).cpu().numpy()
big_Y = torch.vstack(list(Y.values())).cpu().numpy()

In [11]:
print(big_X.shape)
print(big_Y.shape)

(527, 4096)
(527, 4096)


In [12]:
from sklearn.utils import shuffle

big_X_shuffled, big_Y_shuffled = shuffle(big_X, big_Y, random_state=SEED)

In [13]:
# model_lrr = LRR_single(big_X_shuffled, big_Y_shuffled, alpha=1000.0, print_error=True)
model_lrr = LRR_single(big_X_shuffled, big_Y_shuffled, alpha=np.arange(1, 4), print_error=True)

lrr_models = {i: model_lrr for i in hidden_layers}

Running with alpha: [  10.  100. 1000.]
Best lambda: 100.0
Training RMSE: 0.0008, Training R2: 0.9967
Done.


In [14]:
if source_shift:
    with open(f'/scratch/bbjr/skarmakar/neuinv/sk2_items/code/llama8b/source_{orig_p_lang}TO{target_p_lang}-{test_set}.pkl', 'wb') as file:
        pickle.dump(lrr_models, file)
else:
    with open(f'/scratch/bbjr/skarmakar/neuinv/sk2_items/code/llama8b/dest_{orig_p_lang}TO{target_p_lang}-{test_set}.pkl', 'wb') as file:
        pickle.dump(lrr_models, file)

Testing

In [22]:
# p_lang = "python"
p_lang = "c"

task = "Invert a doubly linked list."

sent = 'Write a minimal code in {} language to solve the following task:\n {}'.format(p_lang, task)

messages = [
    {"role": "user", "content": sent}
]

tok_prompts = llm.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True).strip()

out = llm.tokenizer(tok_prompts, return_tensors='pt', padding=True, add_special_tokens=False).to(llm.language_model.device)

outputs = llm.language_model.generate(**out, max_new_tokens=200, do_sample=False)
print(llm.tokenizer.decode(outputs[0], skip_special_tokens=True))

system

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

user

Write a minimal code in c language to solve the following task:
 Invert a doubly linked list.assistant

**Inverting a Doubly Linked List in C**

Here's a minimal code snippet in C to invert a doubly linked list:

```c
#include <stdio.h>
#include <stdlib.h>

// Define the structure for a doubly linked list node
typedef struct Node {
    int data;
    struct Node* next;
    struct Node* prev;
} Node;

// Function to create a new node
Node* createNode(int data) {
    Node* newNode = (Node*) malloc(sizeof(Node));
    if (!newNode) {
        printf("Memory error\n");
        return NULL;
    }
    newNode->data = data;
    newNode->next = NULL;
    newNode->prev = NULL;
    return newNode;
}

// Function to insert a new node at the end of the list
void insertNode(Node** head, int data) {
    Node* newNode = createNode(data);
    if (*head == NULL) {
        *head = newNode;
       


Testing Source Shift

In [19]:
# Loading

orig_p_lang = 'python'
target_p_lang = 'c'
# test_set = ['javascript',]
# test_set = ['php',]
test_set = ['c++',]

with open(f'/scratch/bbjr/skarmakar/neuinv/sk2_items/code/llama8b/source_{orig_p_lang}TO{target_p_lang}-{test_set}.pkl', 'rb') as file:
    lrr_models = pickle.load(file)

In [21]:
# coef = 0.45
# coef = 0.5
# coef = 0.55
# coef = 0.6
coef = 0.65

max_tokens = 200

path = "../all_gitignore/xRFM/code_llama/"

orig_p_lang = 'python'

task = "Invert a doubly linked list"

partial = '''
class Node:
    """Node class for doubly linked list."""
    def __init__(self, data=None):
        self.data = data
        self.next = None
        self.prev = None

class DoublyLinkedList:
    """Doubly linked list class."""
    def __init__(self):
        self.head = None

    def append(self, data):
''' # python

# prompts = ['Write a minimal code in {} language to solve the following task:\n {}'.format(orig_p_lang, task)]
prompts = ['Complete the given partial code to solve the task- {}. \nPartial code:\n{}'.format(task, partial)]


# p = 'javascript'
# p = 'php'
p = 'c++'

p_controller = load_controller_translate(llm, p, orig_p_lang, path=path)
orig_p = p_controller.directions
out = test_concept_vector(p_controller, concept=f"coef: {coef}, ques original: {orig_p_lang}, steered: {p}", prompts=prompts, coef=coef, max_tokens=max_tokens)


new_partial = '''
#include <stdio.h>
#include <stdlib.h>

// Define the structure for a doubly linked list node
typedef struct Node {
    int data;
    struct Node* next;
    struct Node* prev;
} Node;

// Function to create a new node
Node* createNode(int data) {
    Node* newNode = (Node*) malloc(sizeof(Node));
''' # c

# new_prompts = ['Write a minimal code in {} language to solve the following task:\n {}'.format(target_p_lang, task)]
new_prompts = ['Complete the given partial code to solve the task- {}. \nPartial code:\n{}'.format(task, new_partial)]



p1_controller = load_controller_translate(llm, p, target_p_lang, path=path)
orig_p1 = p1_controller.directions
out = test_concept_vector(p1_controller, concept=f"coef: {coef}, ques original: {target_p_lang}, steered: {p}", prompts=new_prompts, coef=coef, max_tokens=max_tokens)


trans_p = apply_auto(orig_p, lrr_models)
p_controller.directions = trans_p

out = test_concept_vector(p_controller, concept=f"coef: {coef}, ques original: {target_p_lang}, source_M({orig_p_lang}to{target_p_lang})(steered): {p}", prompts=new_prompts, coef=coef, max_tokens=max_tokens, orig=False)

Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 8
forward_batch_size   : 16
M_batch_size         : 2048
n_components         : 1

Using xRFM library for RFM-based steering vector extraction
Detector found

========================== No Control ==========================
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

<|eot_id|><|start_header_id|>user<|end_header_id|>

Complete the given partial code to solve the task- Invert a doubly linked list. 
Partial code:

class Node:
    """Node class for doubly linked list."""
    def __init__(self, data=None):
        self.data = data
        self.next = None
        self.prev = None

class DoublyLinkedList:
    """Doubly linked list class."""
    def __init__(self):
        self

In [26]:
coef = 0.8
# coef = 0.85
# coef = 0.9
# coef = 0.95

# trans_p = apply_auto(orig_p, lrr_models)
# p_controller.directions = trans_p

out = test_concept_vector(p_controller, concept=f"coef: {coef}, ques original: {target_p_lang}, source_M({orig_p_lang}to{target_p_lang})(steered): {p}", prompts=new_prompts, coef=coef, max_tokens=max_tokens, orig=False)


========================== + coef: 0.8, ques original: c, source_M(pythontoc)(steered): c++ Control (normal) ==========================
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

<|eot_id|><|start_header_id|>user<|end_header_id|>

Complete the given partial code to solve the task- Invert a doubly linked list. 
Partial code:

#include <stdio.h>
#include <stdlib.h>

// Define the structure for a doubly linked list node
typedef struct Node {
    int data;
    struct Node* next;
    struct Node* prev;
} Node;

// Function to create a new node
Node* createNode(int data) {
    Node* newNode = (Node*) malloc(sizeof(Node));<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Here is the completed code to solve the task:

```csharp
using System;
using System.Linq;

// Define the structure for a doubly linked list node
public class Node
{
    public int Data { get; set; }
    public Node Next { get; set; }
    